In [ ]:
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset
max_seq_length = 2048 # Supports RoPE Scaling internally, so choose any!

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/mistral-7b-bnb-4bit",
    "unsloth/mistral-7b-instruct-v0.2-bnb-4bit",
    "unsloth/llama-2-7b-bnb-4bit",
    "unsloth/gemma-7b-bnb-4bit",
    "unsloth/gemma-7b-it-bnb-4bit", # Instruct version of Gemma 7b
    "unsloth/gemma-2b-bnb-4bit",
    "unsloth/gemma-2b-it-bnb-4bit", # Instruct version of Gemma 2b
    "unsloth/llama-3-8b-bnb-4bit", # [NEW] 15 Trillion token Llama-3
    "unsloth/Phi-3-mini-4k-instruct-bnb-4bit",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-2b-it-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

alpaca_prompt = """Below is an instruction that describes a task with input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN
def formatting_prompts_func(examples):
    instructions = """
Objective: Your task is to generate a sequence of JSON responses to plan robot arm actions based on user input. If the objective cannot be achieved using the provided instructions and available objects, return an error message.

Provide a JSON object containing an array of "actions", identified with the key "actions".

Each action must be represented as an object with appropriate "command" and "parameters".

Available Objects and Coordinates (x,y,z):
1. Purple block = (-86.59, 117.21, -122.30)
2. Yellow block = (-168.94, -129.37, -68)
3. Blue block = (152.76, 158.92, 6)

Available Commands:
1. move: Move the robot arm in a certain direction. Include the "direction" parameter with values "atas" (up), "bawah" (down), "depan" (forward), "belakang" (backward), "kiri" (left), or "kanan" (right).
2. move_to: Move the robot arm to specific coordinates. Include "x", "y", and "z" parameters to specify the target coordinates.
3. suction_cup: Activate or deactivate the suction cup. Use the "action" parameter with values "on" or "off".
4. err_msg: Return an error message if the user's objective cannot be achieved with the current objects and commands. Use the "msg" parameter with value "cannot create action plan with current conditions".

Example Command Usage:
{\"actions\":[{\"command\":\"move\",\"parameters\":{\"direction\":\"atas\"}},{\"command\":\"move_to\",\"parameters\":{\"x\":-30.21,\"y\":233.32,\"z\":-40}},{\"command\":\"suction_cup\",\"parameters\":{\"action\":\"on\"}},{\"command\":\"err_msg\",\"parameters\":{\"msg\":\"cannot create action plan with current conditions\"}}]}"

Usage Instructions:
1. To move an available object to specific coordinates, first activate the suction cup using the "suction_cup" command with "action" set to "on", then move to the object's coordinates using the "move_to" command.
2. Provide placement coordinates for the user's objective using the "move_to" command.
3. To release an object after using the suction cup, deactivate the suction cup first using the "suction_cup" command with "action" set to "off".
4. To move the robot laterally (for example, left, right, forward, backward, up, down), use the "move" command with the appropriate direction.
5. To move an object laterally (for example, left, right, forward, backward, up, down), first move the robot arm to the object's coordinates using the "move_to" command, then use the "move" command with the appropriate direction.
6. If the user's objective cannot be achieved with the current commands and objects, use the "err_msg" command.
    
"""

    inputs       = examples["input"]
    outputs      = examples["response"]
    texts = []
    for input, output in zip(inputs, outputs):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = alpaca_prompt.format(instructions, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }
pass


# Get dataset
dataset = load_dataset("Aryaduta/llm_robot", split = "train")
dataset = dataset.map(formatting_prompts_func, batched = True,)

eval_data = load_dataset("Aryaduta/llm_robot", split = "validation")
eval_data = eval_data.map(formatting_prompts_func, batched = True,)

# Do model patching and add fast LoRA weights
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    max_seq_length = max_seq_length,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

trainer = SFTTrainer(
    model = model,
    train_dataset = dataset,
    eval_dataset = eval_data,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    tokenizer = tokenizer,
    args = TrainingArguments(
        per_device_eval_batch_size=1,
        evaluation_strategy="steps",   
        per_device_train_batch_size = 1,
        eval_steps = 1000,
        warmup_steps = 2,
        num_train_epochs=3.0,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1000,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

In [ ]:
trainer.train()

In [ ]:
# alpaca_prompt = Copied from above
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
inputs = tokenizer(
[
    alpaca_prompt.format(
"""
Objective: Your task is to generate a sequence of JSON responses to plan robot arm actions based on user input. If the objective cannot be achieved using the provided instructions and available objects, return an error message.

Provide a JSON object containing an array of "actions", identified with the key "actions".

Each action must be represented as an object with appropriate "command" and "parameters".

Available Objects and Coordinates (x,y,z):
1. Purple block = (-86.59, 117.21, -122.30)
2. Yellow block = (-168.94, -129.37, -68)
3. Blue block = (152.76, 158.92, 6)

Available Commands:
1. move: Move the robot arm in a certain direction. Include the "direction" parameter with values "atas" (up), "bawah" (down), "depan" (forward), "belakang" (backward), "kiri" (left), or "kanan" (right).
2. move_to: Move the robot arm to specific coordinates. Include "x", "y", and "z" parameters to specify the target coordinates.
3. suction_cup: Activate or deactivate the suction cup. Use the "action" parameter with values "on" or "off".
5. err_msg: Return an error message if the user's objective cannot be achieved with the current objects and commands. Use the "msg" parameter with value "cannot create action plan with current conditions".

Example Command Usage:
{\"actions\":[{\"command\":\"move\",\"parameters\":{\"direction\":\"atas\"}},{\"command\":\"move_to\",\"parameters\":{\"x\":-30.21,\"y\":233.32,\"z\":-40}},{\"command\":\"suction_cup\",\"parameters\":{\"action\":\"on\"}},{\"command\":\"err_msg\",\"parameters\":{\"msg\":\"cannot create action plan with current conditions\"}}]}"

Usage Instructions:
1. To move an available object to specific coordinates, first activate the suction cup using the "suction_cup" command with "action" set to "on", then move to the object's coordinates using the "move_to" command.
2. Provide placement coordinates for the user's objective using the "move_to" command.
3. To release an object after using the suction cup, deactivate the suction cup first using the "suction_cup" command with "action" set to "off".
4. To move the robot laterally (for example, left, right, forward, backward, up, down), use the "move" command with the appropriate direction.
5. To move an object laterally (for example, left, right, forward, backward, up, down), first move the robot arm to the object's coordinates using the "move_to" command, then use the "move" command with the appropriate direction.
6. If the user's objective cannot be achieved with the current commands and objects, use the "err_msg" command.
      
"""
, # instruction
    "move the blue block to the right of the yellow block", # input
    "", # output - leave this blank for generation!
)
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 1024)

In [ ]:
# Decode output and extract JSON response
decoded_output = tokenizer.batch_decode(outputs)[0]

# Find the start and end indices for the JSON response
start_marker = "### Response:"
end_marker = "<eos>"
start_index = decoded_output.find(start_marker) + len(start_marker)
end_index = decoded_output.find(end_marker, start_index)

# Extract the JSON response
json_response = decoded_output[start_index:end_index].strip()

# Print the JSON response
print(json_response)

In [ ]:
# For uploading model to HuggingFace
import os
hf_token = "XXXXXXXX" # Change to your HuggingFace Token

In [ ]:
!huggingface-cli login --token $hf_token

In [ ]:
model.push_to_hub_merged("model_name", tokenizer, save_method = "lora", token = hf_token) # Change your model name to your own liking